# Notebook 01: Data Exploration

Explore the surrogate training data and profile generators interactively.
Run `scripts/02_generate_training_data.py` first to generate the parquet file.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from transformer_lol.utils import get_project_root
from transformer_lol.pv_synthesis import synthesize_pv_profile
from transformer_lol.load_profile import synthesize_fallback_load
from transformer_lol.ambient_profile import synthesize_tropical_ambient
from transformer_lol.thermal_model import TransformerParams, simulate_thermal
from transformer_lol.aging import aging_acceleration_factor, loss_of_life_percent

root = get_project_root()
print('Project root:', root)

In [ ]:
# Load training data
df = pd.read_parquet(root / 'data' / 'processed' / 'surrogate_training_data.parquet')
print(f'Shape: {df.shape}')
print(f'theta_hs range: {df["theta_hs_t"].min():.1f} - {df["theta_hs_t"].max():.1f} degC')
df.describe()

In [ ]:
# Plot one week of profiles for 3 PV penetration levels
n_hours = 168  # 1 week
transformer = TransformerParams()

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

for ax, pv_pen, label in zip(axes, [0.0, 0.30, 0.75], ['0% PV', '30% PV', '75% PV']):
    load = synthesize_fallback_load(n_hours, seed=42)
    pv = synthesize_pv_profile(n_hours, penetration_pu=pv_pen, seed=42)
    net = np.clip(load - pv, 0, 1.5)
    ambient = synthesize_tropical_ambient(n_hours, seed=42)
    result = simulate_thermal(net, ambient, transformer)
    
    ax.plot(net, label='Net load', alpha=0.7)
    ax.plot(result['theta_hs'] / 120, label='theta_HS / 120', alpha=0.7)
    ax.set_ylabel(label)
    ax.legend(loc='upper right', fontsize=8)

axes[-1].set_xlabel('Hour')
plt.tight_layout()
plt.show()